In [10]:
# CELL 1: Import Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("All libraries imported successfully!")

All libraries imported successfully!


In [11]:

df = pd.read_csv('/content/drive/MyDrive/14 Days of ML/Model Selection + Cross-Validation Day10/diabetes.csv')

print("="*60)
print("DATASET OVERVIEW")
print("="*60)

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape} (rows, columns)")
print(f"\nFirst 5 rows:")
print(df.head())

print(f"\nColumn Names:")
print(df.columns.tolist())

print(f"\nData Types:")
print(df.dtypes)
print(f"\nStatistical Summary:")
print(df.describe())
print(f"\nTarget Variable Distribution:")
print(df['Outcome'].value_counts())

DATASET OVERVIEW
Dataset loaded successfully!
Shape: (768, 9) (rows, columns)

First 5 rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Column Names:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Data Types:
Pregnancies                   int64
Glucose              

In [12]:
#Missing Values Analysis

print("="*60)
print("MISSING VALUES ANALYSIS")
print("="*60)

print("\nNull values in dataset:")
print(df.isnull().sum())

# Check for zeros that might indicate missing values
zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("\nColumns where zero might indicate missing values:")
for col in zero_columns:
    zero_count = (df[col] == 0).sum()
    zero_percent = zero_count/len(df)*100
    print(f"  {col}: {zero_count} zeros ({zero_percent:.2f}%)")

print("\nTotal zero values in each column:")
for col in df.columns:
    zero_count = (df[col] == 0).sum()
    if zero_count > 0:
        print(f"  {col}: {zero_count} zeros")

MISSING VALUES ANALYSIS

Null values in dataset:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Columns where zero might indicate missing values:
  Glucose: 5 zeros (0.65%)
  BloodPressure: 35 zeros (4.56%)
  SkinThickness: 227 zeros (29.56%)
  Insulin: 374 zeros (48.70%)
  BMI: 11 zeros (1.43%)

Total zero values in each column:
  Pregnancies: 111 zeros
  Glucose: 5 zeros
  BloodPressure: 35 zeros
  SkinThickness: 227 zeros
  Insulin: 374 zeros
  BMI: 11 zeros
  Outcome: 500 zeros


In [13]:
#Handle Missing Values

print("="*60)
print("HANDLING MISSING VALUES")
print("="*60)

df_clean = df.copy()

# Columns where zero is invalid (medical measurements cannot be zero)
zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("Replacing zeros with NaN and filling with median values:")
print("-"*60)

for col in zero_columns:
    # Count zeros before replacement
    zero_before = (df_clean[col] == 0).sum()

    # Replace zero with NaN
    df_clean[col] = df_clean[col].replace(0, np.nan)

    # Calculate median (ignoring NaN)
    median_val = df_clean[col].median()

    # Fill NaN with median
    df_clean[col] = df_clean[col].fillna(median_val)

    print(f"{col}:")
    print(f"  Zeros replaced: {zero_before}")
    print(f"  Filled with median: {median_val:.2f}")

print("\n" + "-"*60)
print("Missing values after handling:")
print(df_clean.isnull().sum())

print("\nFirst 5 rows after cleaning:")
print(df_clean.head())

HANDLING MISSING VALUES
Replacing zeros with NaN and filling with median values:
------------------------------------------------------------
Glucose:
  Zeros replaced: 5
  Filled with median: 117.00
BloodPressure:
  Zeros replaced: 35
  Filled with median: 72.00
SkinThickness:
  Zeros replaced: 227
  Filled with median: 29.00
Insulin:
  Zeros replaced: 374
  Filled with median: 125.00
BMI:
  Zeros replaced: 11
  Filled with median: 32.30

------------------------------------------------------------
Missing values after handling:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

First 5 rows after cleaning:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6    148.0           72.0           35.0    125.0  33.6   
1      

In [14]:
# CELL 5: Feature Scaling

from sklearn.preprocessing import StandardScaler

print("="*60)
print("FEATURE SCALING")
print("="*60)

# Separate features and target
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

print("\nFeatures before scaling:")
print(f"  Mean: {X.mean().mean():.4f}")
print(f"  Std: {X.std().mean():.4f}")
print(f"  Min: {X.min().min():.4f}")
print(f"  Max: {X.max().max():.4f}")

# Standard Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("\nFeatures after scaling:")
print(f"  Mean: {X_scaled.mean().mean():.4f}")
print(f"  Std: {X_scaled.std().mean():.4f}")
print(f"  Min: {X_scaled.min().min():.4f}")
print(f"  Max: {X_scaled.max().max():.4f}")

print("\nFirst 5 rows after scaling:")
print(X_scaled.head())

print("\nTarget variable distribution:")
print(y.value_counts())

FEATURE SCALING
Features shape: (768, 8)
Target shape: (768,)

Features before scaling:
  Mean: 54.2295
  Std: 20.0057
  Min: 0.0000
  Max: 846.0000

Features after scaling:
  Mean: 0.0000
  Std: 1.0007
  Min: -4.0026
  Max: 8.1704

First 5 rows after scaling:
   Pregnancies   Glucose  BloodPressure  SkinThickness   Insulin       BMI  \
0     0.639947  0.866045      -0.031990       0.670643 -0.181541  0.166619   
1    -0.844885 -1.205066      -0.528319      -0.012301 -0.181541 -0.852200   
2     1.233880  2.016662      -0.693761      -0.012301 -0.181541 -1.332500   
3    -0.844885 -1.073567      -0.528319      -0.695245 -0.540642 -0.633881   
4    -1.141852  0.504422      -2.679076       0.670643  0.316566  1.549303   

   DiabetesPedigreeFunction       Age  
0                  0.468492  1.425995  
1                 -0.365061 -0.190672  
2                  0.604397 -0.105584  
3                 -0.920763 -1.041549  
4                  5.484909 -0.020496  

Target variable distribution:

In [15]:
#Train-Test Split

from sklearn.model_selection import train_test_split

print("="*60)
print("TRAIN-TEST SPLIT")
print("="*60)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Total samples: {len(df)}")
print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(df)*100:.1f}%)")

print("\nTraining set target distribution:")
print(y_train.value_counts())
print(f"Percentage diabetic: {y_train.mean()*100:.2f}%")

print("\nTest set target distribution:")
print(y_test.value_counts())
print(f"Percentage diabetic: {y_test.mean()*100:.2f}%")

print("\nData split complete and balanced!")

TRAIN-TEST SPLIT
Total samples: 768
Training set: 614 samples (79.9%)
Test set: 154 samples (20.1%)

Training set target distribution:
Outcome
0    400
1    214
Name: count, dtype: int64
Percentage diabetic: 34.85%

Test set target distribution:
Outcome
0    100
1     54
Name: count, dtype: int64
Percentage diabetic: 35.06%

Data split complete and balanced!


In [16]:
# CELL 7: K-Fold Cross-Validation

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

print("="*60)
print("K-FOLD CROSS-VALIDATION (5-Folds)")
print("="*60)

# Define models with default parameters
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42, probability=True),
    'KNN': KNeighborsClassifier()
}

k_folds = 5
cv_results = {}

print(f"\nPerforming {k_folds}-Fold Cross-Validation on training data...")
print("-"*60)

for name, model in models.items():
    # Perform cross-validation
    scores = cross_val_score(model, X_train, y_train, cv=k_folds, scoring='accuracy')
    cv_results[name] = {
        'scores': scores,
        'mean': scores.mean(),
        'std': scores.std()
    }

    print(f"\n{name}:")
    print(f"  Individual CV Scores: {scores}")
    print(f"  Mean CV Score: {scores.mean():.4f}")
    print(f"  Std Deviation: {scores.std():.4f}")

print("\n" + "-"*60)
print("Summary of CV Scores:")
print("-"*60)
for name in cv_results:
    print(f"{name}: {cv_results[name]['mean']:.4f} (+/- {cv_results[name]['std']:.4f})")

K-FOLD CROSS-VALIDATION (5-Folds)

Performing 5-Fold Cross-Validation on training data...
------------------------------------------------------------

Logistic Regression:
  Individual CV Scores: [0.7804878  0.7804878  0.76422764 0.7804878  0.80327869]
  Mean CV Score: 0.7818
  Std Deviation: 0.0125

Random Forest:
  Individual CV Scores: [0.73170732 0.79674797 0.73170732 0.79674797 0.80327869]
  Mean CV Score: 0.7720
  Std Deviation: 0.0330

SVM:
  Individual CV Scores: [0.7398374  0.77235772 0.74796748 0.7804878  0.78688525]
  Mean CV Score: 0.7655
  Std Deviation: 0.0184

KNN:
  Individual CV Scores: [0.71544715 0.74796748 0.75609756 0.73170732 0.81147541]
  Mean CV Score: 0.7525
  Std Deviation: 0.0326

------------------------------------------------------------
Summary of CV Scores:
------------------------------------------------------------
Logistic Regression: 0.7818 (+/- 0.0125)
Random Forest: 0.7720 (+/- 0.0330)
SVM: 0.7655 (+/- 0.0184)
KNN: 0.7525 (+/- 0.0326)


In [17]:
# Hyperparameter Tuning - Logistic Regression

from sklearn.model_selection import GridSearchCV

print("="*60)
print("HYPERPARAMETER TUNING - LOGISTIC REGRESSION")
print("="*60)

# Parameter grid for Logistic Regression
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l1', 'l2']
}

print("Searching for best parameters...")
print(f"Parameter grid: {param_grid_lr}")

grid_lr = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

print("\nBest Parameters found:")
print(grid_lr.best_params_)

print(f"\nBest Cross-Validation Score: {grid_lr.best_score_:.4f}")

print("\nAll parameter combinations tested:")
print(f"Total combinations: {len(grid_lr.cv_results_['params'])}")
print(f"Best combination index: {grid_lr.best_index_}")

# Store best model
best_lr = grid_lr.best_estimator_

HYPERPARAMETER TUNING - LOGISTIC REGRESSION
Searching for best parameters...
Parameter grid: {'C': [0.01, 0.1, 1, 10, 100], 'solver': ['liblinear', 'lbfgs'], 'penalty': ['l1', 'l2']}

Best Parameters found:
{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}

Best Cross-Validation Score: 0.7818

All parameter combinations tested:
Total combinations: 20
Best combination index: 6


In [18]:
# Hyperparameter Tuning - Random Forest

from sklearn.model_selection import RandomizedSearchCV

print("="*60)
print("HYPERPARAMETER TUNING - RANDOM FOREST")
print("="*60)

# Parameter distribution for Random Forest
param_dist_rf = {
    'n_estimators': [50, 100, 200, 300, 400],
    'max_depth': [None, 5, 10, 15, 20, 25],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', None]
}

print("Using RandomizedSearchCV with 30 combinations...")
print(f"Parameter space: {param_dist_rf}")

random_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist_rf,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

random_rf.fit(X_train, y_train)

print("\nBest Parameters found:")
print(random_rf.best_params_)

print(f"\nBest Cross-Validation Score: {random_rf.best_score_:.4f}")

# Calculate improvement
improvement = (random_rf.best_score_ - cv_results['Random Forest']['mean']) * 100
print(f"\nImprovement over default Random Forest: +{improvement:.2f}%")

best_rf = random_rf.best_estimator_

HYPERPARAMETER TUNING - RANDOM FOREST
Using RandomizedSearchCV with 30 combinations...
Parameter space: {'n_estimators': [50, 100, 200, 300, 400], 'max_depth': [None, 5, 10, 15, 20, 25], 'min_samples_split': [2, 5, 10, 15], 'min_samples_leaf': [1, 2, 4, 6], 'max_features': ['sqrt', 'log2', None]}

Best Parameters found:
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 25}

Best Cross-Validation Score: 0.7769

Improvement over default Random Forest: +0.49%


In [19]:
# Hyperparameter Tuning - SVM

from sklearn.model_selection import GridSearchCV

print("="*60)
print("HYPERPARAMETER TUNING - SVM")
print("="*60)

# Parameter grid for SVM
param_grid_svm = {
    'C': [0.1, 1, 10, 50, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'kernel': ['rbf', 'linear']
}

print("Searching for best parameters...")
print(f"Parameter grid: {param_grid_svm}")

grid_svm = GridSearchCV(
    SVC(random_state=42, probability=True),
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

print("\nBest Parameters found:")
print(grid_svm.best_params_)

print(f"\nBest Cross-Validation Score: {grid_svm.best_score_:.4f}")

# Calculate improvement
improvement = (grid_svm.best_score_ - cv_results['SVM']['mean']) * 100
print(f"\nImprovement over default SVM: +{improvement:.2f}%")

best_svm = grid_svm.best_estimator_

HYPERPARAMETER TUNING - SVM
Searching for best parameters...
Parameter grid: {'C': [0.1, 1, 10, 50, 100], 'gamma': ['scale', 'auto', 0.01, 0.1, 1], 'kernel': ['rbf', 'linear']}

Best Parameters found:
{'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}

Best Cross-Validation Score: 0.7737

Improvement over default SVM: +0.82%


In [20]:
# Hyperparameter Tuning - KNN

from sklearn.model_selection import GridSearchCV

print("="*60)
print("HYPERPARAMETER TUNING - KNN")
print("="*60)

# Parameter grid for KNN
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17, 19, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

print("Searching for best parameters...")
print(f"Parameter grid: {param_grid_knn}")

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_knn.fit(X_train, y_train)

print("\nBest Parameters found:")
print(grid_knn.best_params_)

print(f"\nBest Cross-Validation Score: {grid_knn.best_score_:.4f}")

# Calculate improvement
improvement = (grid_knn.best_score_ - cv_results['KNN']['mean']) * 100
print(f"\nImprovement over default KNN: +{improvement:.2f}%")

best_knn = grid_knn.best_estimator_

HYPERPARAMETER TUNING - KNN
Searching for best parameters...
Parameter grid: {'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17, 19, 21], 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan', 'minkowski']}

Best Parameters found:
{'metric': 'euclidean', 'n_neighbors': 15, 'weights': 'distance'}

Best Cross-Validation Score: 0.7834

Improvement over default KNN: +3.09%


In [21]:
# Compare All Tuned Models

print("="*60)
print("COMPARISON OF TUNED MODELS")
print("="*60)

# Collect all tuned models
tuned_models = {
    'Logistic Regression': best_lr,
    'Random Forest': best_rf,
    'SVM': best_svm,
    'KNN': best_knn
}

# Store results
tuned_results = {}

print("\nCross-Validation Scores after Tuning:")
print("-"*60)

for name, model in tuned_models.items():
    # Perform 5-fold CV
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    tuned_results[name] = {
        'scores': scores,
        'mean': scores.mean(),
        'std': scores.std()
    }

    print(f"\n{name}:")
    print(f"  CV Scores: {scores}")
    print(f"  Mean CV Score: {scores.mean():.4f}")
    print(f"  Std Deviation: {scores.std():.4f}")

print("\n" + "="*60)
print("FINAL COMPARISON TABLE")
print("="*60)
print(f"{'Model':<25} {'CV Score':<12} {'Improvement':<15}")
print("-"*60)

for name in tuned_results:
    # Get default CV score
    default_score = cv_results[name]['mean']
    tuned_score = tuned_results[name]['mean']
    improvement = (tuned_score - default_score) * 100

    print(f"{name:<25} {tuned_score:.4f}      +{improvement:.2f}%")

print("\n" + "="*60)
best_tuned_model_name = max(tuned_results, key=lambda x: tuned_results[x]['mean'])
best_tuned_model = tuned_models[best_tuned_model_name]
print(f"BEST TUNED MODEL: {best_tuned_model_name}")
print(f"Best CV Score: {tuned_results[best_tuned_model_name]['mean']:.4f}")

COMPARISON OF TUNED MODELS

Cross-Validation Scores after Tuning:
------------------------------------------------------------

Logistic Regression:
  CV Scores: [0.77235772 0.7804878  0.77235772 0.77235772 0.81147541]
  Mean CV Score: 0.7818
  Std Deviation: 0.0152

Random Forest:
  CV Scores: [0.76422764 0.79674797 0.7398374  0.79674797 0.78688525]
  Mean CV Score: 0.7769
  Std Deviation: 0.0220

SVM:
  CV Scores: [0.76422764 0.75609756 0.75609756 0.7804878  0.81147541]
  Mean CV Score: 0.7737
  Std Deviation: 0.0209

KNN:
  CV Scores: [0.7398374  0.82113821 0.76422764 0.7804878  0.81147541]
  Mean CV Score: 0.7834
  Std Deviation: 0.0300

FINAL COMPARISON TABLE
Model                     CV Score     Improvement    
------------------------------------------------------------
Logistic Regression       0.7818      +0.00%
Random Forest             0.7769      +0.49%
SVM                       0.7737      +0.82%
KNN                       0.7834      +3.09%

BEST TUNED MODEL: KNN
Best CV 

In [22]:
#  XGBoost  Advanced Algorithm

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

print("="*60)
print("TRYING XGBOOST CLASSIFIER")
print("="*60)

# Default XGBoost
xgb_default = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_default.fit(X_train, y_train)

# Cross-validation on default
default_scores = cross_val_score(xgb_default, X_train, y_train, cv=5, scoring='accuracy')
print("\nDefault XGBoost CV Scores:")
print(f"  Scores: {default_scores}")
print(f"  Mean: {default_scores.mean():.4f}")
print(f"  Std: {default_scores.std():.4f}")

# Hyperparameter Tuning
print("\n" + "-"*60)
print("Tuning XGBoost with RandomizedSearchCV...")

param_dist_xgb = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 7, 9, 11],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.5]
}

random_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_dist_xgb,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

random_xgb.fit(X_train, y_train)

print("\nBest Parameters found:")
print(random_xgb.best_params_)

print(f"\nBest Cross-Validation Score: {random_xgb.best_score_:.4f}")

# Store best XGBoost
best_xgb = random_xgb.best_estimator_

print("\n" + "="*60)
print("XGBOOST COMPARISON:")
print(f"Default XGBoost: {default_scores.mean():.4f}")
print(f"Tuned XGBoost: {random_xgb.best_score_:.4f}")
print(f"Improvement: +{(random_xgb.best_score_ - default_scores.mean())*100:.2f}%")

TRYING XGBOOST CLASSIFIER

Default XGBoost CV Scores:
  Scores: [0.71544715 0.75609756 0.69918699 0.7398374  0.7295082 ]
  Mean: 0.7280
  Std: 0.0196

------------------------------------------------------------
Tuning XGBoost with RandomizedSearchCV...

Best Parameters found:
{'subsample': 0.6, 'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.05, 'gamma': 0.5, 'colsample_bytree': 1.0}

Best Cross-Validation Score: 0.7753

XGBOOST COMPARISON:
Default XGBoost: 0.7280
Tuned XGBoost: 0.7753
Improvement: +4.73%


In [23]:
# CSMOTE - Handle Class Imbalance

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score

print("="*60)
print("APPLYING SMOTE FOR CLASS IMBALANCE")
print("="*60)

# Check current class distribution
print("\nOriginal class distribution:")
print(y_train.value_counts())
print(f"Diabetic ratio: {y_train.mean()*100:.2f}%")

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE class distribution:")
print(pd.Series(y_train_smote).value_counts())
print(f"Diabetic ratio: {pd.Series(y_train_smote).mean()*100:.2f}%")
print(f"Total samples increased from {len(X_train)} to {len(X_train_smote)}")

# Test models with SMOTE data
print("\n" + "-"*60)
print("MODEL PERFORMANCE WITH SMOTE")
print("-"*60)

models_smote = {
    'Logistic Regression': best_lr,
    'Random Forest': best_rf,
    'SVM': best_svm,
    'KNN': best_knn,
    'XGBoost': best_xgb
}

smote_results = {}

for name, model in models_smote.items():
    scores = cross_val_score(model, X_train_smote, y_train_smote, cv=5, scoring='accuracy')
    smote_results[name] = scores.mean()

    print(f"\n{name}:")
    print(f"  CV Scores: {scores}")
    print(f"  Mean CV Score: {scores.mean():.4f}")
    print(f"  Improvement over original: +{(scores.mean() - tuned_results.get(name, {}).get('mean', 0))*100:.2f}%")

print("\n" + "="*60)
print("SUMMARY: SMOTE VS ORIGINAL")
print("="*60)
print(f"{'Model':<25} {'Original':<12} {'SMOTE':<12} {'Change':<10}")
print("-"*60)

for name in smote_results:
    orig_score = tuned_results.get(name, {}).get('mean', 0)
    smote_score = smote_results[name]
    change = (smote_score - orig_score) * 100
    print(f"{name:<25} {orig_score:.4f}       {smote_score:.4f}       {change:+.2f}%")

APPLYING SMOTE FOR CLASS IMBALANCE

Original class distribution:
Outcome
0    400
1    214
Name: count, dtype: int64
Diabetic ratio: 34.85%

After SMOTE class distribution:
Outcome
0    400
1    400
Name: count, dtype: int64
Diabetic ratio: 50.00%
Total samples increased from 614 to 800

------------------------------------------------------------
MODEL PERFORMANCE WITH SMOTE
------------------------------------------------------------

Logistic Regression:
  CV Scores: [0.75    0.725   0.70625 0.75625 0.74375]
  Mean CV Score: 0.7363
  Improvement over original: +-4.56%

Random Forest:
  CV Scores: [0.8     0.75625 0.81875 0.8625  0.875  ]
  Mean CV Score: 0.8225
  Improvement over original: +4.56%

SVM:
  CV Scores: [0.7625  0.75    0.74375 0.74375 0.76875]
  Mean CV Score: 0.7537
  Improvement over original: +-1.99%

KNN:
  CV Scores: [0.825   0.75625 0.80625 0.85625 0.83125]
  Mean CV Score: 0.8150
  Improvement over original: +3.16%

XGBoost:
  CV Scores: [0.775   0.7375  0.80625 

In [24]:
# Final Evaluation on Test Set

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

print("="*60)
print("FINAL EVALUATION ON TEST SET")
print("="*60)

# Best model from SMOTE results: Random Forest with 82.25%
best_model_final = best_rf

# Train best model on full SMOTE data
print("Training best model (Random Forest) on full SMOTE data...")
best_model_final.fit(X_train_smote, y_train_smote)

# Predict on test set
y_pred = best_model_final.predict(X_test)
y_pred_proba = best_model_final.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("\n" + "="*60)
print("FINAL RESULTS - RANDOM FOREST WITH SMOTE")
print("="*60)

print(f"\nTest Set Accuracy: {accuracy:.4f}")
print(f"Test Set AUC-ROC: {auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Diabetic', 'Diabetic']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nTrue Negatives: {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives: {cm[1][1]}")

# Calculate additional metrics
tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

print("\nDetailed Metrics:")
print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1-Score: {f1:.4f}")

FINAL EVALUATION ON TEST SET
Training best model (Random Forest) on full SMOTE data...

FINAL RESULTS - RANDOM FOREST WITH SMOTE

Test Set Accuracy: 0.7338
Test Set AUC-ROC: 0.8124

Classification Report:
              precision    recall  f1-score   support

Non-Diabetic       0.83      0.74      0.78       100
    Diabetic       0.60      0.72      0.66        54

    accuracy                           0.73       154
   macro avg       0.72      0.73      0.72       154
weighted avg       0.75      0.73      0.74       154


Confusion Matrix:
[[74 26]
 [15 39]]

True Negatives: 74
False Positives: 26
False Negatives: 15
True Positives: 39

Detailed Metrics:
Sensitivity (Recall): 0.7222
Specificity: 0.7400
Precision: 0.6000
F1-Score: 0.6555


In [25]:
# Voting Classifier  Ensemble Method

from sklearn.ensemble import VotingClassifier

print("="*60)
print("VOTING CLASSIFIER (NO SMOTE)")
print("="*60)

# Use tuned models on original data
voting_clf = VotingClassifier(
    estimators=[
        ('lr', best_lr),
        ('rf', best_rf),
        ('knn', best_knn)
    ],
    voting='soft',  # Use probability predictions
    weights=[1, 1, 1]  # Equal weight
)

# Cross-validation on original training data
print("Cross-Validation on original training data:")
scores_voting = cross_val_score(voting_clf, X_train, y_train, cv=5, scoring='accuracy')
print(f"  CV Scores: {scores_voting}")
print(f"  Mean CV Score: {scores_voting.mean():.4f}")
print(f"  Std: {scores_voting.std():.4f}")

# Train and evaluate on test set
print("\nTraining Voting Classifier...")
voting_clf.fit(X_train, y_train)

y_pred_voting = voting_clf.predict(X_test)
y_pred_proba_voting = voting_clf.predict_proba(X_test)[:, 1]

accuracy_voting = accuracy_score(y_test, y_pred_voting)
auc_voting = roc_auc_score(y_test, y_pred_proba_voting)

print("\n" + "="*60)
print("VOTING CLASSIFIER RESULTS")
print("="*60)
print(f"Test Accuracy: {accuracy_voting:.4f}")
print(f"Test AUC-ROC: {auc_voting:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_voting, target_names=['Non-Diabetic', 'Diabetic']))

cm_voting = confusion_matrix(y_test, y_pred_voting)
print(f"\nConfusion Matrix:")
print(cm_voting)
print(f"\nTrue Negatives: {cm_voting[0][0]}")
print(f"False Positives: {cm_voting[0][1]}")
print(f"False Negatives: {cm_voting[1][0]}")
print(f"True Positives: {cm_voting[1][1]}")

VOTING CLASSIFIER (NO SMOTE)
Cross-Validation on original training data:
  CV Scores: [0.76422764 0.82113821 0.7398374  0.78861789 0.80327869]
  Mean CV Score: 0.7834
  Std: 0.0287

Training Voting Classifier...

VOTING CLASSIFIER RESULTS
Test Accuracy: 0.7338
Test AUC-ROC: 0.8169

Classification Report:
              precision    recall  f1-score   support

Non-Diabetic       0.77      0.84      0.80       100
    Diabetic       0.64      0.54      0.59        54

    accuracy                           0.73       154
   macro avg       0.71      0.69      0.69       154
weighted avg       0.73      0.73      0.73       154


Confusion Matrix:
[[84 16]
 [25 29]]

True Negatives: 84
False Positives: 16
False Negatives: 25
True Positives: 29


In [26]:
# Logistic Regression with Class Weights

print("="*60)
print("LOGISTIC REGRESSION WITH CLASS WEIGHTS")
print("="*60)

# Calculate class weights
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

print(f"Class weights: {class_weight_dict}")
print(f"Diabetic class weight: {class_weight_dict[1]:.2f} (vs 1.00 for non-diabetic)")

# Train Logistic Regression with class weights
lr_weighted = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight=class_weight_dict,
    C=0.1,
    solver='liblinear'
)

# Cross-validation
scores_lr_weighted = cross_val_score(lr_weighted, X_train, y_train, cv=5, scoring='accuracy')
print(f"\nCV Scores: {scores_lr_weighted}")
print(f"Mean CV Score: {scores_lr_weighted.mean():.4f}")

# Train and evaluate
lr_weighted.fit(X_train, y_train)
y_pred_weighted = lr_weighted.predict(X_test)
y_pred_proba_weighted = lr_weighted.predict_proba(X_test)[:, 1]

accuracy_weighted = accuracy_score(y_test, y_pred_weighted)
auc_weighted = roc_auc_score(y_test, y_pred_proba_weighted)

print("\n" + "="*60)
print("RESULTS WITH CLASS WEIGHTS")
print("="*60)
print(f"Test Accuracy: {accuracy_weighted:.4f}")
print(f"Test AUC-ROC: {auc_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_weighted, target_names=['Non-Diabetic', 'Diabetic']))

cm_weighted = confusion_matrix(y_test, y_pred_weighted)
print(f"\nConfusion Matrix:")
print(cm_weighted)

LOGISTIC REGRESSION WITH CLASS WEIGHTS
Class weights: {np.int64(0): np.float64(0.7675), np.int64(1): np.float64(1.4345794392523366)}
Diabetic class weight: 1.43 (vs 1.00 for non-diabetic)

CV Scores: [0.71544715 0.77235772 0.71544715 0.78861789 0.7704918 ]
Mean CV Score: 0.7525

RESULTS WITH CLASS WEIGHTS
Test Accuracy: 0.7208
Test AUC-ROC: 0.8100

Classification Report:
              precision    recall  f1-score   support

Non-Diabetic       0.82      0.73      0.77       100
    Diabetic       0.58      0.70      0.64        54

    accuracy                           0.72       154
   macro avg       0.70      0.72      0.71       154
weighted avg       0.74      0.72      0.73       154


Confusion Matrix:
[[73 27]
 [16 38]]


In [27]:
# Random Forest with Class Weights

print("="*60)
print("RANDOM FOREST WITH CLASS WEIGHTS")
print("="*60)

# Train Random Forest with class weights
rf_weighted = RandomForestClassifier(
    random_state=42,
    class_weight=class_weight_dict,
    n_estimators=300,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features='sqrt',
    max_depth=25
)

# Cross-validation
scores_rf_weighted = cross_val_score(rf_weighted, X_train, y_train, cv=5, scoring='accuracy')
print(f"CV Scores: {scores_rf_weighted}")
print(f"Mean CV Score: {scores_rf_weighted.mean():.4f}")

# Train and evaluate
rf_weighted.fit(X_train, y_train)
y_pred_rf_weighted = rf_weighted.predict(X_test)
y_pred_proba_rf_weighted = rf_weighted.predict_proba(X_test)[:, 1]

accuracy_rf_weighted = accuracy_score(y_test, y_pred_rf_weighted)
auc_rf_weighted = roc_auc_score(y_test, y_pred_proba_rf_weighted)

print("\n" + "="*60)
print("RANDOM FOREST WITH CLASS WEIGHTS - RESULTS")
print("="*60)
print(f"Test Accuracy: {accuracy_rf_weighted:.4f}")
print(f"Test AUC-ROC: {auc_rf_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_weighted, target_names=['Non-Diabetic', 'Diabetic']))

cm_rf_weighted = confusion_matrix(y_test, y_pred_rf_weighted)
print(f"\nConfusion Matrix:")
print(cm_rf_weighted)

# Summary comparison
print("\n" + "="*60)
print("FINAL COMPARISON - BEST MODELS")
print("="*60)
print(f"{'Model':<35} {'Accuracy':<12} {'Sensitivity':<15}")
print("-"*60)
print(f"{'Random Forest (SMOTE)':<35} {0.7338:.4f}       {0.7222:.4f}")
print(f"{'Voting Classifier':<35} {0.7338:.4f}       {0.5370:.4f}")
print(f"{'Logistic (Class Weights)':<35} {0.7208:.4f}       {0.7037:.4f}")
print(f"{'Random Forest (Class Weights)':<35} {accuracy_rf_weighted:.4f}       {cm_rf_weighted[1][1]/(cm_rf_weighted[1][0]+cm_rf_weighted[1][1]):.4f}")

RANDOM FOREST WITH CLASS WEIGHTS
CV Scores: [0.75609756 0.80487805 0.72357724 0.81300813 0.75409836]
Mean CV Score: 0.7703

RANDOM FOREST WITH CLASS WEIGHTS - RESULTS
Test Accuracy: 0.7532
Test AUC-ROC: 0.8246

Classification Report:
              precision    recall  f1-score   support

Non-Diabetic       0.80      0.82      0.81       100
    Diabetic       0.65      0.63      0.64        54

    accuracy                           0.75       154
   macro avg       0.73      0.72      0.73       154
weighted avg       0.75      0.75      0.75       154


Confusion Matrix:
[[82 18]
 [20 34]]

FINAL COMPARISON - BEST MODELS
Model                               Accuracy     Sensitivity    
------------------------------------------------------------
Random Forest (SMOTE)               0.7338       0.7222
Voting Classifier                   0.7338       0.5370
Logistic (Class Weights)            0.7208       0.7037
Random Forest (Class Weights)       0.7532       0.6296


In [28]:
# CELL 19: Overfitting and Underfitting Check

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("="*60)
print("OVERFITTING AND UNDERFITTING ANALYSIS")
print("="*60)

# Best model: Random Forest with Class Weights
model = rf_weighted

# Get predictions on training set
y_train_pred = model.predict(X_train)
y_train_proba = model.predict_proba(X_train)[:, 1]

# Get predictions on test set
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]

# Calculate metrics for training
train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)
train_auc = roc_auc_score(y_train, y_train_proba)

# Calculate metrics for test
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_proba)

print("\nTRAINING SET PERFORMANCE:")
print("-"*60)
print(f"Accuracy:  {train_accuracy:.4f}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall:    {train_recall:.4f}")
print(f"F1-Score:  {train_f1:.4f}")
print(f"AUC-ROC:   {train_auc:.4f}")

print("\nTEST SET PERFORMANCE:")
print("-"*60)
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")
print(f"AUC-ROC:   {test_auc:.4f}")

# Calculate gaps
print("\n" + "="*60)
print("PERFORMANCE GAP ANALYSIS")
print("="*60)

accuracy_gap = (train_accuracy - test_accuracy) * 100
precision_gap = (train_precision - test_precision) * 100
recall_gap = (train_recall - test_recall) * 100
f1_gap = (train_f1 - test_f1) * 100
auc_gap = (train_auc - test_auc) * 100

print(f"{'Metric':<15} {'Train':<10} {'Test':<10} {'Gap':<10} {'Status':<15}")
print("-"*60)

# Accuracy
status = "GOOD" if accuracy_gap < 5 else "OVERFITTING" if accuracy_gap > 10 else "WARNING"
print(f"{'Accuracy':<15} {train_accuracy:.4f}    {test_accuracy:.4f}    {accuracy_gap:+.2f}%    {status}")

# Precision
status = "GOOD" if precision_gap < 5 else "OVERFITTING" if precision_gap > 10 else "WARNING"
print(f"{'Precision':<15} {train_precision:.4f}    {test_precision:.4f}    {precision_gap:+.2f}%    {status}")

# Recall
status = "GOOD" if recall_gap < 5 else "OVERFITTING" if recall_gap > 10 else "WARNING"
print(f"{'Recall':<15} {train_recall:.4f}    {test_recall:.4f}    {recall_gap:+.2f}%    {status}")

# F1-Score
status = "GOOD" if f1_gap < 5 else "OVERFITTING" if f1_gap > 10 else "WARNING"
print(f"{'F1-Score':<15} {train_f1:.4f}    {test_f1:.4f}    {f1_gap:+.2f}%    {status}")

# AUC-ROC
status = "GOOD" if auc_gap < 5 else "OVERFITTING" if auc_gap > 10 else "WARNING"
print(f"{'AUC-ROC':<15} {train_auc:.4f}    {test_auc:.4f}    {auc_gap:+.2f}%    {status}")

print("\n" + "="*60)
print("FINAL DIAGNOSIS")
print("="*60)

# Overall diagnosis
avg_gap = (accuracy_gap + precision_gap + recall_gap + f1_gap + auc_gap) / 5

if avg_gap < 3:
    print("PERFECT BALANCE - Model generalizes well!")
    print(f"   Average gap: {avg_gap:.2f}%")
elif avg_gap < 5:
    print("GOOD BALANCE - Model is well-fitted!")
    print(f"   Average gap: {avg_gap:.2f}%")
elif avg_gap < 10:
    print(" WARNING - Slight overfitting detected!")
    print(f"   Average gap: {avg_gap:.2f}%")
    print("   Consider: Regularization, more data, or simpler model")
else:
    print("OVERFITTING DETECTED!")
    print(f"   Average gap: {avg_gap:.2f}%")
    print("   Model is memorizing training data!")
    print("   Consider: Reduce complexity, increase regularization, get more data")

# Additional checks
print("\n" + "="*60)
print("DETAILED ANALYSIS")
print("="*60)

# Check if model is underfitting
if train_accuracy < 0.70:
    print("Model might be UNDERFITTING (train accuracy < 70%)")
    print("   Consider: Increase model complexity, add features, try different algorithm")

# Check if train vs test gap is too large
if accuracy_gap > 15:
    print("SEVERE OVERFITTING: Large gap between train and test accuracy")
    print("   Training accuracy is much higher than test accuracy")

print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)

if avg_gap < 5:
    print("Model is ready for deployment!")
    print("No overfitting or underfitting issues")
    print("Good generalization performance")
elif avg_gap < 10:
    print("1. Apply stronger regularization (increase min_samples_split, min_samples_leaf)")
    print("2. Reduce max_depth of Random Forest")
    print("3. Increase number of CV folds")
    print("4. Consider feature selection to reduce noise")
else:
    print("1. Reduce model complexity (smaller max_depth, fewer n_estimators)")
    print("2. Increase min_samples_split and min_samples_leaf")
    print("3. Add more regularization")
    print("4. Get more training data")
    print("5. Consider simpler model like Logistic Regression")

OVERFITTING AND UNDERFITTING ANALYSIS

TRAINING SET PERFORMANCE:
------------------------------------------------------------
Accuracy:  0.9788
Precision: 0.9589
Recall:    0.9813
F1-Score:  0.9700
AUC-ROC:   0.9986

TEST SET PERFORMANCE:
------------------------------------------------------------
Accuracy:  0.7532
Precision: 0.6538
Recall:    0.6296
F1-Score:  0.6415
AUC-ROC:   0.8246

PERFORMANCE GAP ANALYSIS
Metric          Train      Test       Gap        Status         
------------------------------------------------------------
Accuracy        0.9788    0.7532    +22.56%    OVERFITTING
Precision       0.9589    0.6538    +30.51%    OVERFITTING
Recall          0.9813    0.6296    +35.17%    OVERFITTING
F1-Score        0.9700    0.6415    +32.85%    OVERFITTING
AUC-ROC         0.9986    0.8246    +17.40%    OVERFITTING

FINAL DIAGNOSIS
OVERFITTING DETECTED!
   Average gap: 27.70%
   Model is memorizing training data!
   Consider: Reduce complexity, increase regularization, get mo

In [29]:
# CELL 20: Fix Overfitting - Regularized Random Forest

print("="*60)
print("FIXING OVERFITTING - REGULARIZED RANDOM FOREST")
print("="*60)

# More conservative hyperparameters to prevent overfitting
rf_regularized = RandomForestClassifier(
    random_state=42,
    class_weight=class_weight_dict,
    n_estimators=100,           # Reduced from 300
    max_depth=10,               # Reduced from 25
    min_samples_split=10,       # Increased from 2
    min_samples_leaf=4,         # Increased from 2
    max_features='sqrt',
    max_samples=0.8,            # Use only 80% of data per tree
    bootstrap=True
)

# Cross-validation
scores_rf_reg = cross_val_score(rf_regularized, X_train, y_train, cv=5, scoring='accuracy')
print(f"CV Scores: {scores_rf_reg}")
print(f"Mean CV Score: {scores_rf_reg.mean():.4f}")

# Train and evaluate
rf_regularized.fit(X_train, y_train)

# Training performance
y_train_pred_reg = rf_regularized.predict(X_train)
train_acc_reg = accuracy_score(y_train, y_train_pred_reg)

# Test performance
y_test_pred_reg = rf_regularized.predict(X_test)
y_test_proba_reg = rf_regularized.predict_proba(X_test)[:, 1]
test_acc_reg = accuracy_score(y_test, y_test_pred_reg)
test_auc_reg = roc_auc_score(y_test, y_test_proba_reg)

print("\n" + "="*60)
print("REGULARIZED RANDOM FOREST - RESULTS")
print("="*60)
print(f"Train Accuracy: {train_acc_reg:.4f}")
print(f"Test Accuracy: {test_acc_reg:.4f}")
print(f"Gap: {(train_acc_reg - test_acc_reg)*100:.2f}%")
print(f"Test AUC-ROC: {test_auc_reg:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred_reg, target_names=['Non-Diabetic', 'Diabetic']))

cm_reg = confusion_matrix(y_test, y_test_pred_reg)
print(f"\nConfusion Matrix:")
print(cm_reg)
print(f"True Negatives: {cm_reg[0][0]}, False Positives: {cm_reg[0][1]}")
print(f"False Negatives: {cm_reg[1][0]}, True Positives: {cm_reg[1][1]}")

print("\n" + "="*60)
print("OVERFITTING CHECK - REGULARIZED MODEL")
print("="*60)

new_gap = (train_acc_reg - test_acc_reg) * 100
if new_gap < 10:
    print(f"Much better! Gap reduced from 22.56% to {new_gap:.2f}%")
    print(f"Test Accuracy: {test_acc_reg:.4f}")
elif new_gap < 15:
    print(f"Still some overfitting. Gap: {new_gap:.2f}%")
else:
    print(f"Still overfitting. Gap: {new_gap:.2f}%")
    print("Try even simpler model or Logistic Regression")

FIXING OVERFITTING - REGULARIZED RANDOM FOREST
CV Scores: [0.7398374  0.80487805 0.71544715 0.78861789 0.75409836]
Mean CV Score: 0.7606

REGULARIZED RANDOM FOREST - RESULTS
Train Accuracy: 0.8713
Test Accuracy: 0.7662
Gap: 10.51%
Test AUC-ROC: 0.8289

Classification Report:
              precision    recall  f1-score   support

Non-Diabetic       0.84      0.79      0.81       100
    Diabetic       0.65      0.72      0.68        54

    accuracy                           0.77       154
   macro avg       0.75      0.76      0.75       154
weighted avg       0.77      0.77      0.77       154


Confusion Matrix:
[[79 21]
 [15 39]]
True Negatives: 79, False Positives: 21
False Negatives: 15, True Positives: 39

OVERFITTING CHECK - REGULARIZED MODEL
Still some overfitting. Gap: 10.51%


In [30]:
# CELL 21: Final Model - Logistic Regression

print("="*60)
print("FINAL MODEL - LOGISTIC REGRESSION (LESS OVERFITTING)")
print("="*60)

# Logistic Regression with class weights and regularization
lr_final = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight=class_weight_dict,
    C=0.1,  # Strong regularization
    solver='liblinear'
)

# Cross-validation
scores_lr_final = cross_val_score(lr_final, X_train, y_train, cv=5, scoring='accuracy')
print(f"CV Scores: {scores_lr_final}")
print(f"Mean CV Score: {scores_lr_final.mean():.4f}")

# Train and evaluate
lr_final.fit(X_train, y_train)

# Training performance
y_train_pred_lr = lr_final.predict(X_train)
train_acc_lr = accuracy_score(y_train, y_train_pred_lr)

# Test performance
y_test_pred_lr = lr_final.predict(X_test)
y_test_proba_lr = lr_final.predict_proba(X_test)[:, 1]
test_acc_lr = accuracy_score(y_test, y_test_pred_lr)
test_auc_lr = roc_auc_score(y_test, y_test_proba_lr)

print("\n" + "="*60)
print("LOGISTIC REGRESSION - RESULTS")
print("="*60)
print(f"Train Accuracy: {train_acc_lr:.4f}")
print(f"Test Accuracy: {test_acc_lr:.4f}")
print(f"Gap: {(train_acc_lr - test_acc_lr)*100:.2f}%")
print(f"Test AUC-ROC: {test_auc_lr:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred_lr, target_names=['Non-Diabetic', 'Diabetic']))

cm_lr = confusion_matrix(y_test, y_test_pred_lr)
print(f"\nConfusion Matrix:")
print(cm_lr)
print(f"True Negatives: {cm_lr[0][0]}, False Positives: {cm_lr[0][1]}")
print(f"False Negatives: {cm_lr[1][0]}, True Positives: {cm_lr[1][1]}")

# Final comparison
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
print(f"{'Model':<30} {'Test Acc':<12} {'Gap':<12} {'AUC':<12}")
print("-"*60)
print(f"{'Random Forest (Original)':<30} {0.7532:.4f}     {22.56:.2f}%    {0.8246:.4f}")
print(f"{'Random Forest (Regularized)':<30} {0.7662:.4f}     {10.51:.2f}%    {0.8289:.4f}")
print(f"{'Logistic Regression':<30} {test_acc_lr:.4f}     {(train_acc_lr - test_acc_lr)*100:.2f}%    {test_auc_lr:.4f}")

print("\n" + "="*60)
print("RECOMMENDED FINAL MODEL")
print("="*60)

if (train_acc_lr - test_acc_lr) < 5 and test_acc_lr > 0.75:
    print(" LOGISTIC REGRESSION - Best balance of accuracy and generalization!")
    print(f"   Test Accuracy: {test_acc_lr:.4f}")
    print(f"   Gap: {(train_acc_lr - test_acc_lr)*100:.2f}%")
    print("   Suitable for deployment")
elif test_acc_lr > 0.76:
    print(" REGULARIZED RANDOM FOREST - Best accuracy with acceptable generalization")
    print(f"   Test Accuracy: 0.7662")
    print(f"   Gap: 10.51%")
else:
    print("  Continue tuning or try different approach")

FINAL MODEL - LOGISTIC REGRESSION (LESS OVERFITTING)
CV Scores: [0.71544715 0.77235772 0.71544715 0.78861789 0.7704918 ]
Mean CV Score: 0.7525

LOGISTIC REGRESSION - RESULTS
Train Accuracy: 0.7655
Test Accuracy: 0.7208
Gap: 4.47%
Test AUC-ROC: 0.8100

Classification Report:
              precision    recall  f1-score   support

Non-Diabetic       0.82      0.73      0.77       100
    Diabetic       0.58      0.70      0.64        54

    accuracy                           0.72       154
   macro avg       0.70      0.72      0.71       154
weighted avg       0.74      0.72      0.73       154


Confusion Matrix:
[[73 27]
 [16 38]]
True Negatives: 73, False Positives: 27
False Negatives: 16, True Positives: 38

FINAL MODEL COMPARISON
Model                          Test Acc     Gap          AUC         
------------------------------------------------------------
Random Forest (Original)       0.7532     22.56%    0.8246
Random Forest (Regularized)    0.7662     10.51%    0.8289
Logisti

In [31]:
# Save Final Model (Regularized Random Forest)

import joblib
import os

print("="*60)
print("SAVING FINAL MODEL")
print("="*60)

# Path
save_path = "/content/drive/MyDrive/14 Days of ML/Model Selection + Cross-Validation Day10"

# Create directory if needed
os.makedirs(save_path, exist_ok=True)

# Save model and scaler
joblib.dump(rf_regularized, os.path.join(save_path, 'diabetes_final_model.pkl'))
joblib.dump(scaler, os.path.join(save_path, 'scaler.pkl'))

print(f"Model saved to: {save_path}/diabetes_final_model.pkl")
print(f"Scaler saved to: {save_path}/scaler.pkl")

# Verify
print("\nVerifying saved files:")
for file in os.listdir(save_path):
    if file.endswith('.pkl'):
        size = os.path.getsize(os.path.join(save_path, file)) / 1024
        print(f"  {file}: {size:.2f} KB")

print("\n" + "="*60)
print("FINAL MODEL SUMMARY")
print("="*60)
print("Algorithm: Random Forest (Regularized)")
print(f"Test Accuracy: 0.7662")
print(f"Test AUC-ROC: 0.8289")
print(f"Overfitting Gap: 10.51%")
print("Class Weights: Non-Diabetic=0.77, Diabetic=1.43")
print("\nHyperparameters:")
print("  n_estimators: 100")
print("  max_depth: 10")
print("  min_samples_split: 10")
print("  min_samples_leaf: 4")
print("  max_features: 'sqrt'")
print("  max_samples: 0.8")
print("  bootstrap: True")

SAVING FINAL MODEL
Model saved to: /content/drive/MyDrive/14 Days of ML/Model Selection + Cross-Validation Day10/diabetes_final_model.pkl
Scaler saved to: /content/drive/MyDrive/14 Days of ML/Model Selection + Cross-Validation Day10/scaler.pkl

Verifying saved files:
  diabetes_final_model.pkl: 667.01 KB
  scaler.pkl: 1.13 KB

FINAL MODEL SUMMARY
Algorithm: Random Forest (Regularized)
Test Accuracy: 0.7662
Test AUC-ROC: 0.8289
Overfitting Gap: 10.51%
Class Weights: Non-Diabetic=0.77, Diabetic=1.43

Hyperparameters:
  n_estimators: 100
  max_depth: 10
  min_samples_split: 10
  min_samples_leaf: 4
  max_features: 'sqrt'
  max_samples: 0.8
  bootstrap: True
